# Lara's notities

## Preprocessen/sample methoden
The pipeline described by MNE is https://mne.tools/stable/documentation/cookbook.html is VERY detailed. I think our data is already quite polished maybe? They only talk about downsampling and normalization in the project description so maybe such things are more complicated than needed. 
Do downsampling with basic factor 10 right now and do average pooling, factor can be bigger when slow. PCA for dimension reduction yayy
Z-score normalization can be better then min-max when amplitudes vary a lot for the MEG data (z-score uses the normal distribution while min-max is linear, https://www.codecademy.com/article/min-max-zscore-normalization).

## (a) Modellen om uit te kiezen
MNE tools are made for MEG analysis and visualization. Also learning? sklearn. Could do normal classification too maybe. I chose 1D CNN becauseee it seems fine (survey: https://www.sciencedirect.com/science/article/pii/S0888327020307846).

## (b) Test framework voor zowel cross als intra testen

## (c) Hyperparameter tweaking framework


We moeten vooral ook beschrijven hoe we verbeteren enzo. Miss een soort labjournaal bijhouden?

In [ ]:
%pip install torch numpy scipy scikit-learn matplotlib h5py

# Read data

In [ ]:
import os
import glob
import h5py
import numpy as np

from sklearn.

# Functions to load/get/extract adminstrative stuff
def get_dataset_name(file_name_with_dir):
    file_name_without_dir = file_name_with_dir.split("/")[-1]
    temp = file_name_without_dir.split(".")[:-1]
    dataset_name = "".join(temp)
    return dataset_name

def load_h5_file(filepath):
    f = h5py.File(filepath, 'r')
    dataset_name = get_dataset_name(filepath)
    return f.get(dataset_name)[()]      # Outputs matrix that is the numpy array of the data in the h5py file

def get_label(filepath):    # extract classification activity label: 0="rest", 1="math", 2="memory" en 3="motor"
    name = os.path.basename(filepath)
    if name.startswith("rest"):     # maybe slow sorry, idk how else to do this
        return 0
    elif name.startswith("math"):
        return 1
    elif name.startswith("memory"):
        return 2
    elif name.startswith("motor"):
        return 3

# downsample when loading data
def downsample(matrix, factor=10):
    nsensors, ntimesteps = matrix.shape
    down_ntimesteps = ntimesteps // factor
    matrix = matrix[:, :down_ntimesteps * factor]
    matrix = matrix.reshape(nsensors,down_ntimesteps,factor).mean(axis=2)
    return matrix

# extract useful properties
def extract_properties(matrix):
    mean = np.mean(matrix, axis=1)
    std = np.std(matrix, axis=1)
    max_value = np.max(matrix, axis=1)
    min_value = np.min(matrix, axis=1)
    return np.concatenate([mean,std,min_value,max_value])

# load a folder of data!
def load_folder(folder):
    x = []
    y = []
    files = glob.glob("data/" + folder + "/*.h5")
    for file in files:
        matrix = load_h5_file(file)
        matrix = downsample(matrix)
        props = extract_properties(matrix)
        x.append(props)
        y.append(get_label(file))
    return np.array(x), np.array(y)

# Load all the data (maybe requires too much memory)
x_train_intra, y_train_intra = load_folder("Intra/train")
x_test_intra, y_test_intra = load_folder("Intra/test")

x_train_cross, y_train_cross = load_folder("Cross/train")
x_test_cross1, y_test_cross1 = load_folder("Cross/test1")
x_test_cross2, y_test_cross2 = load_folder("Cross/test2")
x_test_cross3, y_test_cross3 = load_folder("Cross/test3")